In [7]:

import requests
import os
import time
import numpy as np
from pathlib import Path
from bs4 import BeautifulSoup
from typing import List, Optional, Any, Dict
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_core.language_models.llms import LLM
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate 
from langchain_core.output_parsers import StrOutputParser  
from langchain_core.runnables import RunnablePassthrough  
from pydantic import BaseModel, Field
from PyPDF2 import PdfReader




In [8]:
# Remove proxy settings (you're off-campus)
for key in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY']:
    if key in os.environ:
        del os.environ[key]
        print(f"✓ Removed {key}")

# Verify they're gone
print("\nProxy settings after cleanup:")
print(f"http_proxy: {os.environ.get('http_proxy', 'Not set')}")
print(f"https_proxy: {os.environ.get('https_proxy', 'Not set')}")


✓ Removed http_proxy
✓ Removed https_proxy

Proxy settings after cleanup:
http_proxy: Not set
https_proxy: Not set


In [9]:
# HKBU API CONFIGURATION 
api_key = "f55f10bc-9d4f-4751-b7d7-ac519834c8e8"
base_url = "https://genai.hkbu.edu.hk/api/v0/rest"

# Model configurations with their correct API versions
MODEL_CONFIGS = {
    "gpt": {
        "name": "gpt-5-mini",
        "api_version": "2024-12-01-preview"
    },
    "deepseek": {
        "name": "deepseek",
        "api_version": "2024-05-01-preview"
    },
    "gemini": {
        "name": "gemini",
        "api_version": "Latest"
    },
    "llama": {
        "name": "llama",
        "api_version": "20240723"
    },
    "qwen3-max": {
        "name": "qwen3-max",
        "api_version": "v1"
    },
    "qwen-plus": {
        "name": "qwen-plus",
        "api_version": "v1"
    }
}

# Choose the model 
SELECTED_MODEL = "qwen3-max"  # use qwen3-max for best overall performance
model_name = MODEL_CONFIGS[SELECTED_MODEL]["name"]
api_version = MODEL_CONFIGS[SELECTED_MODEL]["api_version"]

# Embeddings configuration
embedding_api_version = "2024-05-01-preview"
embedding_model = "text-embedding-3-small"

print(f"Selected Model: {model_name}")
print(f"Model API Version: {api_version}")
print(f"Embedding Model: {embedding_model}")
print(f"Embedding API Version: {embedding_api_version}")


Selected Model: qwen3-max
Model API Version: v1
Embedding Model: text-embedding-3-small
Embedding API Version: 2024-05-01-preview


In [10]:
# EMBEDDINGS WRAPPER
class HKBUEmbeddings(Embeddings):
    api_key: str = Field(default="")
    base_url: str = Field(default="")
    model: str = Field(default="")
    api_version: str = Field(default="")
    
    def __init__(self, api_key: str, base_url: str, model: str, api_version: str):
        super().__init__()
        self.api_key = api_key
        self.base_url = base_url
        self.model = model
        self.api_version = api_version
        # Build embeddings URL with api_version
        self.url = f"{base_url}/deployments/{model}/embeddings?api-version={api_version}"
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Embed a list of documents"""
        embeddings = []
        headers = {
            "accept": "application/json",
            "Content-Type": "application/json",
            "api-key": self.api_key,
        }
        
        for text in texts:
            payload = {"input": text}
            response = requests.post(self.url, json=payload, headers=headers)
            
            if response.status_code == 200:
                embeddings.append(response.json()["data"][0]["embedding"])
            else:
                raise Exception(f"Embedding failed: {response.status_code} - {response.text}")
        
        return embeddings
    
    def embed_query(self, text: str) -> List[float]:
        """Embed a single query"""
        return self.embed_documents([text])[0]


In [11]:
class HKBULLM(LLM):
    api_key: str = Field(default="")
    base_url: str = Field(default="")
    model: str = Field(default="qwen")
    api_version: str = Field(default="2024-12-01-preview")
    temperature: float = Field(default=0.4)
    max_tokens: int = Field(default=400)
    
    @property
    def _llm_type(self) -> str:
        return "hkbu"
    
    def _call(
        self, 
        prompt: str, 
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any
    ) -> str:
        """Call HKBU API"""
        url = f"{self.base_url}/deployments/{self.model}/chat/completions?api-version={self.api_version}"
        headers = {
            "accept": "application/json",
            "Content-Type": "application/json",
            "api-key": self.api_key,
        }
        
        system_message = (
            "You are a helpful Java programming tutor. Provide clear, accurate explanations "
            "with code examples when appropriate. Break down complex concepts into simple steps."
        )
        
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ]
        
        payload = {
            "messages": messages,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
            "top_p": 1,
            "stream": False
        }
        
        response = requests.post(url, json=payload, headers=headers)
        
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        else:
            raise Exception(f"API call failed: {response.status_code} - {response.text}")

In [5]:
# JAVA CONTENT SCRAPER
PYPDF2_AVAILABLE = True
def safe_filename(name):
    """Convert string to safe filename"""
    return "".join([c if c.isalnum() or c in (' ', '-', '_') else '_' for c in name]).strip()

def scrape_oracle_docs():
    """Scrape Oracle Java Documentation - BRIEF"""
    print("\n📚 Scraping Oracle Java Documentation...")
    
    # Oracle Java Tutorial is now redirected to dev.java
    # We'll scrape the Java SE documentation pages
    
    os.makedirs('./java_docs/oracle', exist_ok=True)
    
    topics = [
        ("https://docs.oracle.com/javase/tutorial/java/concepts/index.html", "oop_concepts"),
        ("https://docs.oracle.com/javase/tutorial/java/javaOO/index.html", "classes_objects"),
        ("https://docs.oracle.com/javase/tutorial/java/IandI/index.html", "interfaces_inheritance"),
        ("https://docs.oracle.com/javase/tutorial/collections/index.html", "collections"),
        ("https://docs.oracle.com/javase/tutorial/essential/exceptions/index.html", "exceptions"),
        ("https://docs.oracle.com/javase/tutorial/essential/concurrency/index.html", "concurrency"),
    ]
    
    saved = 0
    
    for url, name in topics:
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Oracle docs use different content divs
            content = (soup.find('div', id='PageContent') or 
                      soup.find('div', class_='MainFlow') or
                      soup.find('main'))
            
            if content:
                for script in content(["script", "style", "nav"]):
                    script.decompose()
                
                text = content.get_text(separator='\n', strip=True)
                
                if len(text) > 300:
                    with open(f'./java_docs/oracle/oracle_{saved:03d}_{name}.txt', 'w', encoding='utf-8') as f:
                        f.write(f"Source: Oracle Java Documentation\nTopic: {name}\nURL: {url}\n\n{text}")
                    saved += 1
                    time.sleep(1)
        except:
            pass
    
    print(f"  ✅ Saved {saved} pages")

def scrape_exception_handling():
    """Target exception handling content"""
    print("\n📚 Scraping Exception Handling Content...")
    os.makedirs('./java_docs/exceptions', exist_ok=True)
    
    sources = [
        ("https://www.geeksforgeeks.org/exceptions-in-java/", "gfg_exceptions"),
        ("https://www.geeksforgeeks.org/checked-vs-unchecked-exceptions-in-java/", "gfg_checked_unchecked"),
        ("https://www.geeksforgeeks.org/flow-control-in-try-catch-finally-in-java/", "gfg_try_catch"),
        ("https://www.w3schools.com/java/java_try_catch.asp", "w3_try_catch"),
        ("https://www.javatpoint.com/exception-handling-in-java", "jtp_exceptions"),
    ]
    
    saved = 0
    for url, name in sources:
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            content = (soup.find('article') or soup.find('div', id='main') or 
                      soup.find('div', class_='onlycontent'))
            
            if content:
                for script in content(["script", "style"]):
                    script.decompose()
                text = content.get_text(separator='\n', strip=True)
                
                if len(text) > 300:
                    with open(f'./java_docs/exceptions/{name}.txt', 'w', encoding='utf-8') as f:
                        f.write(f"Source: {url}\nTopic: Exception Handling\n\n{text}")
                    saved += 1
                    time.sleep(1)
        except:
            pass
    
    print(f"  ✅ Saved {saved} files")


#  W3SCHOOLS JAVA TUTORIAL
def scrape_w3schools():
    """Scrape W3Schools Java tutorials - BRIEF OUTPUT"""
    print("\n📚 Scraping W3Schools Java Tutorial...")
    
    base = "https://www.w3schools.com/java/"
    start_url = f"{base}default.asp"
    
    os.makedirs('./java_docs/w3schools', exist_ok=True)
    
    try:
        response = requests.get(start_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        sidebar = soup.find('div', id='leftmenuinnerinner')
        
        if not sidebar:
            print("⚠️ Could not find W3Schools sidebar")
            return
        
        links = sidebar.find_all('a', href=True)
        saved_count = 0
        seen_urls = set()
        
        for link in links[:100]:
            href = link['href']
            
            if 'java' not in href.lower() and not href.endswith('.asp'):
                continue
            
            if href.startswith('http'):
                page_url = href
            elif href.startswith('/'):
                page_url = "https://www.w3schools.com" + href
            else:
                page_url = base + href
            
            if page_url in seen_urls:
                continue
            seen_urls.add(page_url)
            
            title = safe_filename(link.text.strip())
            if not title or len(title) < 3:
                continue
            
            try:
                page_response = requests.get(page_url, timeout=10)
                page_soup = BeautifulSoup(page_response.text, 'html.parser')
                content = page_soup.find('div', id='main')
                
                if content:
                    for script in content(["script", "style", "nav"]):
                        script.decompose()
                    
                    text_content = content.get_text(separator='\n', strip=True)
                    
                    if len(text_content) > 200:
                        filename = f'./java_docs/w3schools/w3_{saved_count:03d}_{title}.txt'
                        with open(filename, 'w', encoding='utf-8') as f:
                            f.write(f"Source: W3Schools Java Tutorial\n")
                            f.write(f"Title: {link.text}\n")
                            f.write(f"URL: {page_url}\n\n")
                            f.write(text_content)
                        
                        saved_count += 1
                        time.sleep(0.5)
                
            except Exception as e:
                pass  # Silent fail
        
        print(f"  ✅ Saved {saved_count} pages")
                
    except Exception as e:
        print(f"  ✗ Failed: {e}")


# Apply same pattern to all scrapers
def scrape_geeksforgeeks():
    """Scrape GeeksforGeeks - BRIEF"""
    print("\n📚 Scraping GeeksforGeeks...")
    os.makedirs('./java_docs/geeksforgeeks', exist_ok=True)
    
    topics = [
        "java-oops-concepts", "classes-objects-java", "constructors-in-java",
        "inheritance-in-java", "polymorphism-in-java", "encapsulation-in-java",
        "abstraction-in-java", "interfaces-in-java", "java-exception-handling",
        "collections-in-java-2", "arraylist-in-java", "hashmap-in-java",
        "multithreading-in-java", "java-io-tutorial", "java-strings",
        "java-generics", "java-lambda-expressions", "stream-in-java",
        "java-packages", "java-access-modifiers"
    ]
    
    base = "https://www.geeksforgeeks.org"
    saved = 0
    
    for topic in topics:
        try:
            response = requests.get(f"{base}/{topic}/")
            soup = BeautifulSoup(response.text, 'html.parser')
            article = soup.find('div', class_='text') or soup.find('article')
            
            if article:
                for script in article(["script", "style"]):
                    script.decompose()
                text = article.get_text(separator='\n', strip=True)
                
                with open(f'./java_docs/geeksforgeeks/gfg_{saved:03d}_{topic}.txt', 'w', encoding='utf-8') as f:
                    f.write(f"Source: GeeksforGeeks\nTopic: {topic}\nURL: {base}/{topic}/\n\n{text}")
                saved += 1
                time.sleep(1)
        except:
            pass
    
    print(f"  ✅ Saved {saved} articles")


def download_think_java():
    """Download Think Java - BRIEF"""
    print("\n📚 Downloading Think Java...")
    os.makedirs('./java_docs/books', exist_ok=True)
    
    try:
        response = requests.get("https://greenteapress.com/thinkjava7/thinkjava2.pdf", stream=True)
        if response.status_code == 200:
            with open('./java_docs/books/think_java.pdf', 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            if PYPDF2_AVAILABLE:
                reader = PdfReader('./java_docs/books/think_java.pdf')
                full_text = [page.extract_text() for page in reader.pages]
                with open('./java_docs/books/think_java_full.txt', 'w', encoding='utf-8') as f:
                    f.write("Source: Think Java\nURL: https://greenteapress.com/wp/think-java-2e/\n\n")
                    f.write('\n'.join(full_text))
                print(f"  ✅ Extracted {len(reader.pages)} pages")
            else:
                print("  ⚠️ PDF saved, but PyPDF2 not installed for extraction")
    except Exception as e:
        print(f"  ✗ Failed: {e}")

# INTRODUCTION TO PROGRAMMING USING JAVA
def scrape_javanotes():
    """Scrape Introduction to Programming Using Java by David Eck"""
    print("\n📚 Scraping Introduction to Programming Using Java...")
    
    base_url = "http://math.hws.edu/javanotes9/"
    os.makedirs('./java_docs/javanotes', exist_ok=True)
    
    # Main chapters
    chapters = [
        "c1-overview/index.html",
        "c2-basics/index.html",
        "c3-control/index.html",
        "c4-subroutines/index.html",
        "c5-OOP/index.html",
        "c6-arrays-arraylists/index.html",
        "c7-recursion/index.html",
        "c8-correctness-robustness/index.html",
        "c9-threads/index.html",
        "c10-generics-streams/index.html",
        "c11-io-files-networking/index.html",
        "c12-gui/index.html"
    ]
    
    for chapter in chapters:
        url = base_url + chapter
        chapter_name = chapter.split('/')[0]
        
        try:
            response = requests.get(url)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Extract main content
            content = soup.find('div', class_='content')
            if not content:
                content = soup.find('body')
            
            if content:
                for script in content(["script", "style", "nav"]):
                    script.decompose()
                
                text = content.get_text(separator='\n', strip=True)
                
                filename = f'./java_docs/javanotes/{chapter_name}.txt'
                with open(filename, 'w', encoding='utf-8') as f:
                    f.write(f"Source: Introduction to Programming Using Java by David Eck\n")
                    f.write(f"Chapter: {chapter_name}\n")
                    f.write(f"URL: {url}\n\n")
                    f.write(text)
                
                print(f"  ✓ Saved: {chapter_name}")
                time.sleep(0.5)
                
        except Exception as e:
            print(f"  ✗ Error on {chapter_name}: {e}")

# ============================================
# OPEN DATA STRUCTURES (IN JAVA)

def scrape_open_data_structures():
    """Scrape Open Data Structures in Java"""
    print("\n📚 Scraping Open Data Structures (Java)...")
    
    base_url = "https://opendatastructures.org/ods-java/"
    os.makedirs('./java_docs/data_structures', exist_ok=True)
    
    # Main chapters
    chapters = [
        "ods-java-node1.html",  # Introduction
        "ods-java-node2.html",  # Array-Based Lists
        "ods-java-node3.html",  # Linked Lists
        "ods-java-node4.html",  # Skiplists
        "ods-java-node5.html",  # Hash Tables
        "ods-java-node6.html",  # Binary Trees
        "ods-java-node7.html",  # Random Binary Search Trees
        "ods-java-node8.html",  # Scapegoat Trees
        "ods-java-node9.html",  # Red-Black Trees
        "ods-java-node10.html", # Heaps
        "ods-java-node11.html", # Sorting
        "ods-java-node12.html"  # Graphs
    ]
    
    for i, chapter in enumerate(chapters):
        url = base_url + chapter
        
        try:
            response = requests.get(url)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Extract main content
            content = soup.find('div', class_='main')
            if not content:
                content = soup.find('body')
            
            if content:
                for script in content(["script", "style"]):
                    script.decompose()
                
                text = content.get_text(separator='\n', strip=True)
                
                filename = f'./java_docs/data_structures/ods_chapter_{i+1:02d}.txt'
                with open(filename, 'w', encoding='utf-8') as f:
                    f.write(f"Source: Open Data Structures (in Java)\n")
                    f.write(f"URL: {url}\n\n")
                    f.write(text)
                
                print(f"  ✓ Saved: Chapter {i+1}")
                time.sleep(0.5)
                
        except Exception as e:
            print(f"  ✗ Error on chapter {i+1}: {e}")


# TO COLLECT ALL CONTENT

def create_comprehensive_java_content():
    """Collect Java content from all sources"""
    
    print("="*60)
    print("COMPREHENSIVE JAVA CONTENT COLLECTION")
    print("="*60)
    
    os.makedirs('./java_docs', exist_ok=True)
    
  
    scrape_w3schools()
    scrape_oracle_docs()
    scrape_geeksforgeeks()
    scrape_exception_handling() 
    download_think_java()
    scrape_javanotes()
    scrape_open_data_structures()
    
    # Count total files
    total_files = sum([len(files) for r, d, files in os.walk('./java_docs')])
    
    print("\n" + "="*60)
    print(f"✅ COLLECTION COMPLETE!")
    print(f"Total files collected: {total_files}")
    print("="*60)
    
    # Show directory structure
    print("\nDirectory structure:")
    for root, dirs, files in os.walk('./java_docs'):
        level = root.replace('./java_docs', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')


if __name__ == "__main__":
    create_comprehensive_java_content()
    pass


COMPREHENSIVE JAVA CONTENT COLLECTION

📚 Scraping W3Schools Java Tutorial...
  ✅ Saved 87 pages

📚 Scraping Oracle Java Documentation...
  ✅ Saved 6 pages

📚 Scraping GeeksforGeeks...
  ✅ Saved 11 articles

📚 Scraping Exception Handling Content...
  ✅ Saved 1 files

📚 Downloading Think Java...
  ✅ Extracted 366 pages

📚 Scraping Introduction to Programming Using Java...
  ✓ Saved: c1-overview
  ✓ Saved: c2-basics
  ✓ Saved: c3-control
  ✓ Saved: c4-subroutines
  ✓ Saved: c5-OOP
  ✓ Saved: c6-arrays-arraylists
  ✓ Saved: c7-recursion
  ✓ Saved: c8-correctness-robustness
  ✓ Saved: c9-threads
  ✓ Saved: c10-generics-streams
  ✓ Saved: c11-io-files-networking
  ✓ Saved: c12-gui

📚 Scraping Open Data Structures (Java)...
  ✓ Saved: Chapter 1
  ✓ Saved: Chapter 2
  ✓ Saved: Chapter 3
  ✓ Saved: Chapter 4
  ✓ Saved: Chapter 5
  ✓ Saved: Chapter 6
  ✓ Saved: Chapter 7
  ✓ Saved: Chapter 8
  ✓ Saved: Chapter 9
  ✓ Saved: Chapter 10
  ✓ Saved: Chapter 11
  ✓ Saved: Chapter 12

✅ COLLECTION COMP

In [12]:
# ============================================
# MODULAR RAG SYSTEM FUNCTIONS
# ============================================

from sentence_transformers import CrossEncoder

def load_or_create_vectorstore(chunks, embeddings, vectorstore_path="./java_rag_vectorstore_comprehensive"):
    """Load existing vectorstore or create new one with checkpoints"""
    
    # Load existing partial vectorstore
    if os.path.exists(vectorstore_path):
        print("✅ Loading partial vectorstore to continue...")
        vectorstore = FAISS.load_local(
            vectorstore_path,
            embeddings,
            allow_dangerous_deserialization=True
        )
        
        # Check how many chunks are already indexed
        current_count = vectorstore.index.ntotal
        print(f"Current vectorstore has {current_count} vectors")
        
        # Calculate remaining chunks to process
        remaining_chunks = chunks[current_count:] if current_count < len(chunks) else []
        
        if remaining_chunks:
            print(f"Resuming: {len(remaining_chunks)} chunks remaining")
            # Continue from checkpoint
            batch_size = 10  # Even smaller batches
            
            for i in range(0, len(remaining_chunks), batch_size):
                batch = remaining_chunks[i:i+batch_size]
                print(f" Processing batch {i//batch_size + 1}...")
                
                try:
                    vectorstore.add_documents(batch)
                    vectorstore.save_local(vectorstore_path)
                    print(f" ✅ Saved ({current_count + i + len(batch)} total)")
                    time.sleep(5)  # Longer delay
                except Exception as e:
                    if "403" in str(e) or "429" in str(e):
                        print(f" ⏸️ Rate limit hit. Waiting 60 seconds...")
                        time.sleep(60)
                        continue
                    raise
        else:
            print("✅ Vectorstore is complete!")
        
        return vectorstore

    
    # Create new vectorstore with batching AND RATE LIMITING
    print(f"Creating vectorstore from {len(chunks)} chunks...")
    batch_size = 20 
    vectorstore = None
    
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        batch_num = (i // batch_size) + 1
        total_batches = (len(chunks) + batch_size - 1) // batch_size
        
        print(f"  Batch {batch_num}/{total_batches} ({len(batch)} chunks)...")
        
        try:
            if vectorstore is None:
                vectorstore = FAISS.from_documents(batch, embeddings)
            else:
                vectorstore.add_documents(batch)
            
            vectorstore.save_local(vectorstore_path)
            print(f"  ✅ Saved ({i+len(batch)}/{len(chunks)} chunks)")
            
            # INCREASED DELAY from 1s to 3s
            time.sleep(3)  # Give API time to breathe
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            print(f"  Last checkpoint: {i} chunks")
            
            # If 403 error, wait longer and retry
            if "403" in str(e):
                print("  Waiting 10 seconds due to rate limit...")
                time.sleep(10)
                # Try to continue from last checkpoint
                continue
            else:
                raise
    
    print("✅ Vectorstore created!")
    return vectorstore









def build_rag_chain(vectorstore, llm, k=5):
    """Build LCEL RAG chain with HYBRID retrieval - ORIGINAL BASELINE"""
    
    # HYBRID RETRIEVAL: MMR for diversity
    retriever = vectorstore.as_retriever(
        search_type="mmr",  # Maximum Marginal Relevance
        search_kwargs={
            "k": k,
            "fetch_k": 15,      # Fetch 15 candidates
            "lambda_mult": 0.7  # Balance: 0.7 relevance + 0.3 diversity
        }
    )
    
    # ORIGINAL PROMPT (the one that worked)
    template = """You are a Java programming tutor. Answer ONLY using the context below.

STRICT RULES:
1. Every sentence MUST come from the context
2. If context lacks info, say: "My knowledge base doesn't fully cover this topic"
3. NEVER use external knowledge
4. Copy code examples EXACTLY from context
5. Keep answers under 200 words
6. Start with a direct 1-sentence definition

Context:
{context}

Question: {question}

Answer (context-only, max 200 words):"""
    
    prompt = PromptTemplate.from_template(template)
    
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    return rag_chain, retriever



def load_documents(docs_dir='./java_docs'):
    """Load all .txt documents from directory"""
    print("Loading documents...")
    all_docs = []
    
    for root, dirs, files in os.walk(docs_dir):
        for file in files:
            if file.endswith('.txt'):
                filepath = os.path.join(root, file)
                try:
                    loader = TextLoader(filepath, encoding='utf-8')
                    docs = loader.load()
                    all_docs.extend(docs)
                except Exception as e:
                    print(f"  ⚠️ Could not load {file}: {e}")
    
    print(f"✅ Loaded {len(all_docs)} documents")
    return all_docs


def chunk_documents(docs, chunk_size=600, chunk_overlap=150):
    """Split documents into chunks"""
    print("Chunking documents...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(docs)
    print(f"✅ Created {len(chunks)} chunks")
    return chunks


# ============================================
# MAIN SETUP FUNCTION
# ============================================

def setup_rag_system(rebuild_vectorstore=False, force_delete=False):
    """Main function to set up RAG system"""
    
    # Initialize models
    print("Initializing embeddings and LLM...")
    embeddings = HKBUEmbeddings(
        api_key=api_key,
        base_url=base_url,
        model=embedding_model,
        api_version=embedding_api_version
    )
    
    llm = HKBULLM(
        api_key=api_key,
        base_url=base_url,
        model=model_name,
        api_version=api_version,
        temperature=1,
        max_tokens=400,
    )
    print("Models initialized")
    
    vectorstore_path = "./java_rag_vectorstore_comprehensive"
    
    # Only delete if explicitly requested
    if force_delete and os.path.exists(vectorstore_path):
        import shutil
        shutil.rmtree(vectorstore_path)
        print("Deleted old vectorstore")
    
    # Load existing or create/resume
    if os.path.exists(vectorstore_path) and not rebuild_vectorstore:
        print("Loading existing vectorstore (skipping document processing)...")
        vectorstore = FAISS.load_local(
            vectorstore_path,
            embeddings,
            allow_dangerous_deserialization=True
        )
    else:
        # Process documents (will resume if partial vectorstore exists)
        docs = load_documents()
        chunks = chunk_documents(docs)
        vectorstore = load_or_create_vectorstore(chunks, embeddings)
    
    # Build RAG chain
    print("Building RAG chain...")
    rag_chain, retriever = build_rag_chain(vectorstore, llm, k=3)
    print("RAG system ready!")
    
    return rag_chain, retriever




# ============================================
# QUERY FUNCTION
# ============================================

def query_rag(rag_chain, retriever, question, show_sources=True):
    """Query the RAG system and optionally show sources"""
    
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    
    # Get answer
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")
    
    # Show sources
    if show_sources:
        print("\nRetrieved Sources:")
        docs = retriever.invoke(question)
        for i, doc in enumerate(docs[:3], 1):
            source = doc.metadata.get('source', 'Unknown')
            print(f"  {i}. {source[:80]}...")
    
    return answer



rag_chain, retriever = setup_rag_system(rebuild_vectorstore=False)

# Example usage
#query_rag(rag_chain, retriever, "What is inheritance in Java?")
#query_rag(rag_chain, retriever, "How do I create an ArrayList?")
#query_rag(rag_chain, retriever, "Explain polymorphism with an example")


Initializing embeddings and LLM...
Models initialized
Loading existing vectorstore (skipping document processing)...
Building RAG chain...
RAG system ready!


In [16]:
# ============================================
# CELL: EVALUATION WITH NLI FAITHFULNESS (FIXED)
# ============================================
from typing import Dict, List
import numpy as np
import time
import re
import os
import glob
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

# ============================================
# SET OFFLINE MODE
# ============================================
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# Disable BERTScore to avoid roberta-large download
USE_BERTSCORE = False  # Set to False to skip BERTScore

# ============================================
# FIND LOCAL MODEL PATHS
# ============================================
print("Finding local model paths...")

# MiniLM path
minilm_path = glob.glob(os.path.expanduser('~/models/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/*'))[0]

# NLI Cross-Encoder path
nli_path = os.path.expanduser('~/models/nli-deberta-xsmall')

print(f"MiniLM path: {minilm_path}")
print(f"NLI model path: {nli_path}")

# ============================================
# LOAD MODELS FROM LOCAL CACHE
# ============================================
print("\nLoading evaluation models from local cache...")
_sbert_model = SentenceTransformer(minilm_path, local_files_only=True)
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# Load NLI Cross-Encoder (different from pipeline)
_nli_model = CrossEncoder(nli_path, max_length=512)

print("✓ Models loaded from local cache\n")

# ============================================
# NLI-BASED FAITHFULNESS
# ============================================

def decompose_into_claims(answer: str) -> List[str]:
    """Split answer into atomic claims"""
    if not answer or len(answer.strip()) == 0:
        return []
    sentences = re.split(r'[.!?]+', answer)
    claims = [s.strip() for s in sentences if len(s.strip()) > 10]
    return claims

def check_claim_entailment(claim: str, context: str) -> Dict[str, float]:
    """Check if context entails claim using NLI Cross-Encoder"""
    try:
        # Cross-encoder expects pairs of [premise, hypothesis]
        # Context is premise, claim is hypothesis
        score = _nli_model.predict([(context, claim)])[0]
        
        # Cross-encoder NLI models output 3 scores: [contradiction, neutral, entailment]
        # We want the entailment score (index 2)
        if isinstance(score, (list, np.ndarray)) and len(score) == 3:
            entailment_score = score[2]  # entailment is usually last
        else:
            # If single score, treat as entailment probability
            entailment_score = float(score)
        
        return {'entailment': entailment_score}
    except Exception as e:
        print(f"Error in NLI: {e}")
        return {'entailment': 0.5}

def calculate_nli_faithfulness(answer: str, context: str) -> Dict:
    """TRUE FAITHFULNESS via NLI entailment"""
    claims = decompose_into_claims(answer)
    
    if not claims:
        return {'faithfulness_score': 1.0, 'num_claims': 0, 'entailed_claims': 0}
    
    entailed_count = 0
    for claim in claims:
        nli_result = check_claim_entailment(claim, context)
        if nli_result['entailment'] > 0.5:
            entailed_count += 1
    
    return {
        'faithfulness_score': entailed_count / len(claims),
        'num_claims': len(claims),
        'entailed_claims': entailed_count
    }

def calculate_semantic_faithfulness(answer: str, context: str) -> float:
    """Semantic similarity without BERTScore"""
    # Use only SBERT and ROUGE (skip BERTScore to avoid download)
    emb1 = _sbert_model.encode(answer)
    emb2 = _sbert_model.encode(context)
    sbert_sim = util.cos_sim(emb1, emb2).item()
    
    rouge_scores = _rouge.score(context, answer)
    rouge_f1 = rouge_scores['rougeL'].fmeasure
    
    # Weighted average (60% SBERT, 40% ROUGE since no BERTScore)
    weighted_avg = (0.6 * sbert_sim + 0.4 * rouge_f1)
    return weighted_avg

def calculate_relevance(docs, ground_truth: str) -> float:
    """Context recall"""
    context = " ".join([doc.page_content for doc in docs]).lower()
    gt_words = set(ground_truth.lower().split())
    if not gt_words:
        return 1.0
    matches = sum(1 for word in gt_words if word in context)
    return matches / len(gt_words)

# Rest of your evaluation code stays the same...



# ============================================
# EVALUATION FUNCTION
# ============================================

def evaluate_rag_system(rag_chain, retriever, test_questions: List[Dict], use_nli: bool = True):
    """Evaluate with BOTH NLI and semantic faithfulness"""
    
    results = {
        "nli_faithfulness": [],
        "semantic_faithfulness": [],
        "context_recall": [],
        "response_time": [],
        "claim_details": []
    }
    
    for idx, item in enumerate(test_questions, 1):
        question = item["question"]
        ground_truth = item.get("ground_truth", "")
        
        print(f"\n[{idx}/{len(test_questions)}] {question[:50]}...")
        
        start = time.time()
        answer = rag_chain.invoke(question)
        docs = retriever.invoke(question)
        response_time = time.time() - start
        results["response_time"].append(response_time)
        
        context_text = " ".join([doc.page_content for doc in docs])
        
        # NLI Faithfulness (TRUE)
        if use_nli:
            nli_result = calculate_nli_faithfulness(answer, context_text)
            nli_faith = nli_result['faithfulness_score']
            results["nli_faithfulness"].append(nli_faith)
            results["claim_details"].append(nli_result)
            print(f"  ✓ NLI Faithfulness: {nli_faith:.2%} ({nli_result['entailed_claims']}/{nli_result['num_claims']} claims)")
        
        # Semantic Faithfulness (OLD)
        sem_faith = calculate_semantic_faithfulness(answer, context_text)
        results["semantic_faithfulness"].append(sem_faith)
        print(f"  ✓ Semantic: {sem_faith:.2%}")
        
        # Context recall
        if ground_truth:
            relevance = calculate_relevance(docs, ground_truth)
            results["context_recall"].append(relevance)
            print(f"  ✓ Context Recall: {relevance:.2%}")
        
        print(f"  ✓ Time: {response_time:.2f}s")
    
    return results

# ============================================
# RUN EVALUATION
# ============================================

test_dataset = [
    {"question": "What is inheritance in Java?", "ground_truth": "inheritance extends class subclass superclass reusability"},
    {"question": "How do I create an ArrayList?", "ground_truth": "ArrayList import java.util new add"},
    {"question": "Explain polymorphism with an example", "ground_truth": "polymorphism method overriding overloading compile runtime"},
    {"question": "What are Java interfaces?", "ground_truth": "interface implements abstract methods multiple inheritance"},
    {"question": "How does exception handling work?", "ground_truth": "try catch throw throws exception finally"},
    {"question": "What is encapsulation in Java?", "ground_truth": "encapsulation private public getter setter access control"}
]

print("\n" + "="*70)
print("EVALUATING RAG SYSTEM WITH NLI FAITHFULNESS")
print("="*70 + "\n")

# Run evaluation
results = evaluate_rag_system(rag_chain, retriever, test_dataset, use_nli=True)

# Print summary
print("\n" + "="*70)
print("RAG SYSTEM PERFORMANCE SUMMARY")
print("="*70)

if results["nli_faithfulness"]:
    nli_scores = results["nli_faithfulness"]
    print(f"\n📊 TRUE FAITHFULNESS (NLI):")
    print(f"   Average: {np.mean(nli_scores):.2%}")
    print(f"   Min: {min(nli_scores):.2%}")
    print(f"   Max: {max(nli_scores):.2%}")
    
    total_claims = sum(d['num_claims'] for d in results['claim_details'])
    total_entailed = sum(d['entailed_claims'] for d in results['claim_details'])
    print(f"\n   Total Claims: {total_claims}")
    print(f"   Entailed: {total_entailed} ({total_entailed/total_claims*100:.1f}%)")

sem_scores = results["semantic_faithfulness"]
print(f"\n📊 SEMANTIC SIMILARITY (old):")
print(f"   Average: {np.mean(sem_scores):.2%}")

if results["context_recall"]:
    print(f"\n📊 CONTEXT RECALL:")
    print(f"   Average: {np.mean(results['context_recall']):.2%}")

print(f"\n⏱️ RESPONSE TIME:")
print(f"   Average: {np.mean(results['response_time']):.2f}s")
print(f"   Min: {min(results['response_time']):.2f}s")
print(f"   Max: {max(results['response_time']):.2f}s")

print("\n" + "="*70)

# Store for later
eval_results = results


Finding local model paths...
MiniLM path: /home/comp/f2231870/models/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
NLI model path: /home/comp/f2231870/models/nli-deberta-xsmall

Loading evaluation models from local cache...
✓ Models loaded from local cache


EVALUATING RAG SYSTEM WITH NLI FAITHFULNESS


[1/6] What is inheritance in Java?...
  ✓ NLI Faithfulness: 85.71% (6/7 claims)
  ✓ Semantic: 74.33%
  ✓ Context Recall: 50.00%
  ✓ Time: 7.58s

[2/6] How do I create an ArrayList?...
  ✓ NLI Faithfulness: 100.00% (8/8 claims)
  ✓ Semantic: 74.28%
  ✓ Context Recall: 100.00%
  ✓ Time: 8.04s

[3/6] Explain polymorphism with an example...
  ✓ NLI Faithfulness: 100.00% (8/8 claims)
  ✓ Semantic: 72.73%
  ✓ Context Recall: 83.33%
  ✓ Time: 5.74s

[4/6] What are Java interfaces?...
  ✓ NLI Faithfulness: 100.00% (9/9 claims)
  ✓ Semantic: 92.71%
  ✓ Context Recall: 100.00%
  ✓ Time: 5.15s

[5/6] How does exception handling work?...
  ✓ NLI 

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", 
                       "rouge-score", "bert-score", "--quiet"])
print("✅ Installed")

In [2]:
# ============================================================
# PRODUCTION RAG EVALUATION — NLI FAITHFULNESS (v2)
# Uses current rag_system.py: all-MiniLM-L6-v2 + RoutedRetriever
# ============================================================

import sys, os, re, time, json
import numpy as np
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

# ── 1. LOAD PRODUCTION RAG SYSTEM ───────────────────────────
sys.path.insert(0, '.')   # ensure repo root is on path
from rag_system import setup_rag_system

print("Loading production RAG system...")
rag_chain, retriever = setup_rag_system(rebuild_vectorstore=False)
print("✅ Production RAG system ready\n")

# ── 2. LOAD NLI MODEL ───────────────────────────────────────
# Try local cache first (HKBU server), then fall back to HuggingFace download
NLI_LOCAL = os.path.expanduser('~/models/nli-deberta-xsmall')
NLI_HF    = 'cross-encoder/nli-deberta-v3-small'

nli_model_path = NLI_LOCAL if os.path.exists(NLI_LOCAL) else NLI_HF
print(f"Loading NLI model from: {nli_model_path}")
_nli_model  = CrossEncoder(nli_model_path, max_length=512)
_sbert      = SentenceTransformer('all-MiniLM-L6-v2')
_rouge      = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
print("✅ Evaluation models loaded\n")

# ── 3. TEST QUESTIONS (20 queries covering all major topics) ─
TEST_QUESTIONS = [
    # Core Java concepts
    {"question": "What is inheritance in Java?"},
    {"question": "How do I create an ArrayList?"},
    {"question": "Explain polymorphism with an example"},
    {"question": "What are Java interfaces?"},
    {"question": "How does exception handling work?"},
    {"question": "What is encapsulation in Java?"},
    # New questions for production evaluation
    {"question": "What is the difference between == and .equals() in Java?"},
    {"question": "How does a HashMap work in Java?"},
    {"question": "What is a NullPointerException and how do I fix it?"},
    {"question": "Explain the difference between abstract class and interface"},
    {"question": "What does the static keyword mean in Java?"},
    {"question": "How do I read a file in Java?"},
    {"question": "What is method overloading versus overriding?"},
    {"question": "How does garbage collection work in Java?"},
    {"question": "What is a StackOverflowError?"},
    {"question": "Explain try-catch-finally in Java"},
    {"question": "What is the Java Collections Framework?"},
    {"question": "How do lambda expressions work in Java?"},
    {"question": "What is a constructor in Java?"},
    {"question": "How does the for-each loop work in Java?"},
    {"question": "How do I join a classroom on CodeTutor?"},
    {"question": "How does the quiz feature work?"},
    {"question": "What is the AI tutor on the platform?"},
    {"question": "How do I take a practical test?"},
    {"question": "What is the roadmap feature?"},
    # At the end of TEST_QUESTIONS list, add 25 more:
    {"question": "What is a Java generic type?"},
    {"question": "How does ArrayList differ from LinkedList?"},
    {"question": "What is the difference between int and Integer?"},
    {"question": "How do I sort a list in Java?"},
    {"question": "What is a checked vs unchecked exception?"},
    {"question": "How does String immutability work in Java?"},
    {"question": "What is method chaining in Java?"},
    {"question": "How do I implement Comparable in Java?"},
    {"question": "What is an enum in Java?"},
    {"question": "How does instanceof work?"},
    {"question": "What is a functional interface in Java?"},
    {"question": "How does Optional work in Java?"},
    {"question": "What is the difference between List and Set?"},
    {"question": "How do I handle multiple exceptions?"},
    {"question": "What is a varargs method?"},
    {"question": "How do synchronized methods work?"},
    {"question": "What is autoboxing and unboxing?"},
    {"question": "How do I use the ternary operator?"},
    {"question": "What is a nested class in Java?"},
    {"question": "How do I convert String to int?"},
    {"question": "How do I save my work on CodeTutor?"},
    {"question": "Can I view my quiz history?"},
    {"question": "How does the highlight-to-ask feature work?"},
    {"question": "What does the progress dashboard show?"},
    {"question": "How do I switch between Basic and Enhanced Java tracks?"},
]

# ── 4. NLI HELPER FUNCTIONS ─────────────────────────────────
def decompose_claims(answer):
    sentences = re.split(r'[.!?]+', answer)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def nli_faithfulness(answer, context):
    claims = decompose_claims(answer)
    if not claims:
        return {"score": 1.0, "num_claims": 0, "entailed": 0}
    entailed = 0
    for claim in claims:
        raw = _nli_model.predict([(context, claim)])[0]
        score = raw[2] if (isinstance(raw, (list, np.ndarray)) and len(raw) == 3) else float(raw)
        if score > 0.5:
            entailed += 1
    return {"score": entailed / len(claims), "num_claims": len(claims), "entailed": entailed}

def semantic_sim(answer, context):
    e1 = _sbert.encode(answer)
    e2 = _sbert.encode(context)
    sbert_sim  = util.cos_sim(e1, e2).item()
    rouge_f1   = _rouge.score(context, answer)['rougeL'].fmeasure
    return round(0.6 * sbert_sim + 0.4 * rouge_f1, 4)

# ── 5. RUN EVALUATION ────────────────────────────────────────
print("=" * 70)
print("PRODUCTION RAG EVALUATION — NLI FAITHFULNESS")
print(f"Model: all-MiniLM-L6-v2 + RoutedRetriever + qwen3-max")
print("=" * 70 + "\n")

results = []

for idx, item in enumerate(TEST_QUESTIONS, 1):
    question = item["question"]
    print(f"[{idx}/{len(TEST_QUESTIONS)}] {question}")

    start = time.time()
    answer, docs = rag_chain(question)   # production chain returns (answer, docs)
    elapsed = round(time.time() - start, 2)

    context = " ".join(doc.page_content for doc in docs)

    nli    = nli_faithfulness(answer, context)
    sem    = semantic_sim(answer, context)

    results.append({
        "id":           idx,
        "question":     question,
        "answer":       answer,
        "nli_score":    round(nli["score"] * 100, 2),
        "num_claims":   nli["num_claims"],
        "entailed":     nli["entailed"],
        "semantic":     round(sem * 100, 2),
        "response_time": elapsed,
        "num_docs":     len(docs),
    })

    print(f"  ✓ NLI Faithfulness : {nli['score']*100:.2f}%  ({nli['entailed']}/{nli['num_claims']} claims)")
    print(f"  ✓ Semantic Sim     : {sem*100:.2f}%")
    print(f"  ✓ Time             : {elapsed}s\n")

# ── 6. SUMMARY ───────────────────────────────────────────────
nli_scores  = [r["nli_score"]    for r in results]
sem_scores  = [r["semantic"]     for r in results]
times       = [r["response_time"] for r in results]
tot_claims  = sum(r["num_claims"] for r in results)
tot_entail  = sum(r["entailed"]   for r in results)

print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"\n📊 NLI FAITHFULNESS:")
print(f"   Average : {np.mean(nli_scores):.2f}%")
print(f"   Min     : {np.min(nli_scores):.2f}%")
print(f"   Max     : {np.max(nli_scores):.2f}%")
print(f"   Total claims    : {tot_claims}")
print(f"   Entailed claims : {tot_entail}  ({tot_entail/tot_claims*100:.1f}%)")

print(f"\n📊 SEMANTIC SIMILARITY:")
print(f"   Average : {np.mean(sem_scores):.2f}%")

print(f"\n⏱️  RESPONSE TIME:")
print(f"   Average : {np.mean(times):.2f}s")
print(f"   Min     : {np.min(times):.2f}s")
print(f"   Max     : {np.max(times):.2f}s")

# ── 7. SAVE TO JSON ──────────────────────────────────────────
os.makedirs("evaluation", exist_ok=True)
output = {
    "metadata": {
        "embedding_model":  "sentence-transformers/all-MiniLM-L6-v2",
        "llm":              "qwen3-max (HKBU GenAI Platform)",
        "retriever":        "RoutedRetriever (MMR, java_knowledge + platform_guide)",
        "nli_model":        nli_model_path,
        "num_queries":      len(results),
        "total_claims":     tot_claims,
        "entailed_claims":  tot_entail,
        "timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
    },
    "aggregate": {
        "nli_avg":    round(np.mean(nli_scores), 2),
        "nli_min":    round(np.min(nli_scores), 2),
        "nli_max":    round(np.max(nli_scores), 2),
        "semantic_avg": round(np.mean(sem_scores), 2),
        "time_avg":   round(np.mean(times), 2),
        "time_min":   round(np.min(times), 2),
        "time_max":   round(np.max(times), 2),
    },
    "results": results
}

with open("evaluation/nli_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"\n✅ Results saved to evaluation/nli_results.json")
print(f"   Commit this file to make the Appendix B reference valid.")

Loading production RAG system...
Initializing FAISS RAG system...
   LLM Model: qwen3-max
   Embedding Model: sentence-transformers/all-MiniLM-L6-v2 (384-dim, offline)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17095.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ LocalEmbeddings initialized (384-dim, offline, balanced quality/speed)
   Note: First run will download model (~420MB), then cached locally
✅ HKBU LLM initialized for text generation
Preparing java_knowledge vectorstore...
✅ Loaded java_knowledge vectorstore with 3010 vectors
Preparing platform_guide vectorstore...
✅ Loaded platform_guide vectorstore with 69 vectors
Building RAG chain...
✅ FAISS RAG system ready!
   Active vectorstores: 2
   Retrieval mode: routed-split

✅ Production RAG system ready

Loading NLI model from: cross-encoder/nli-deberta-v3-small


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 9779.51it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11911.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Evaluation models loaded

PRODUCTION RAG EVALUATION — NLI FAITHFULNESS
Model: all-MiniLM-L6-v2 + RoutedRetriever + qwen3-max

[1/50] What is inheritance in Java?
🔀 Routed retrieval: java-first for query 'What is inheritance in Java?' -> 3 docs
  ✓ NLI Faithfulness : 100.00%  (7/7 claims)
  ✓ Semantic Sim     : 75.28%
  ✓ Time             : 5.71s

[2/50] How do I create an ArrayList?
🔀 Routed retrieval: java-first for query 'How do I create an ArrayList?' -> 3 docs
  ✓ NLI Faithfulness : 11.11%  (1/9 claims)
  ✓ Semantic Sim     : 72.92%
  ✓ Time             : 6.11s

[3/50] Explain polymorphism with an example
🔀 Routed retrieval: java-first for query 'Explain polymorphism with an example' -> 3 docs
  ✓ NLI Faithfulness : 100.00%  (11/11 claims)
  ✓ Semantic Sim     : 90.72%
  ✓ Time             : 8.4s

[4/50] What are Java interfaces?
🔀 Routed retrieval: java-first for query 'What are Java interfaces?' -> 3 docs
  ✓ NLI Faithfulness : 87.50%  (7/8 claims)
  ✓ Semantic Sim     : 82.28%